<a href="https://colab.research.google.com/github/theNewMKE/kitten_tts_Nano_quantization_compare/blob/main/Kitten_tts_quantization_compare_INT8_FP32.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 😻 Kitten TTS Model Quantizatin Comparison FP32 & INT8




The Kitten TTS model can be found here:
https://github.com/KittenML/KittenTTS

Model comparison and information:
| Model | Parameters | File Size | Quantization | HuggingFace |
|-------|-----------|-----------|-------------|-------------|
| **Nano int8** ⭐ | 15M | **~20 MB** | INT8 | [kitten-tts-nano-0.8-int8](https://huggingface.co/KittenML/kitten-tts-nano-0.8-int8) |
| **Nano fp32** | 15M | ~50 MB | FP32 | [kitten-tts-nano-0.8-fp32](https://huggingface.co/KittenML/kitten-tts-nano-0.8-fp32) |
| **Micro** | 40M | ~40 MB | — | [kitten-tts-micro-0.8](https://huggingface.co/KittenML/kitten-tts-micro-0.8) |
| **Mini** | 80M | ~79 MB | — | [kitten-tts-mini-0.8](https://huggingface.co/KittenML/kitten-tts-mini-0.8) |

⭐ **Nano int8 is under 25 MB** — the smallest model in the lineup, great for edge / mobile deployment.

**Available voices:** `Bella`, `Jasper`, `Luna`, `Bruno`, `Rosie`, `Hugo`, `Kiki`, `Leo`

[Follow this YouTube video to download all four models](https://www.youtube.com/watch?v=YpQWdrfzSzQ)


[Steps 1-4 are taken from this colab](https://dripl.ink/3BPIH)

## Here we focuse on how to diagnose the accuracy drop or performance loss (if any) between FP32 and INT8 models.

## 1.Install KittenTTS

In [16]:
!pip install -q https://github.com/KittenML/KittenTTS/releases/download/0.8/kittentts-0.8.0-py3-none-any.whl
!pip install -q soundfile

## 2.Load All Three Models

We load each model and measure how long it takes to download + initialize.

In [17]:
import time
from kittentts import KittenTTS

models = {}
model_configs = {
    "Nano int8 (15M params, ~20MB) ⭐": "KittenML/kitten-tts-nano-0.8-int8",
    "Nano fp32 (15M params, ~50MB)": "KittenML/kitten-tts-nano-0.8-fp32",
    "Micro (40M params, ~40MB)": "KittenML/kitten-tts-micro-0.8",
    "Mini (80M params, ~79MB)": "KittenML/kitten-tts-mini-0.8",
}

for label, model_id in model_configs.items():
    print(f"⏳ Loading {label}...")
    t0 = time.time()
    models[label] = KittenTTS(model_id)
    print(f"   ✅ Loaded in {time.time() - t0:.1f}s\n")

print("All models ready!")

⏳ Loading Nano int8 (15M params, ~20MB) ⭐...


   ✅ Loaded in 4.6s

⏳ Loading Nano fp32 (15M params, ~50MB)...
   ✅ Loaded in 2.1s

⏳ Loading Micro (40M params, ~40MB)...
   ✅ Loaded in 1.5s

⏳ Loading Mini (80M params, ~79MB)...
   ✅ Loaded in 1.6s

All models ready!


In [18]:
# check if the models are downloaded
!ls -la /root/.cache/huggingface/hub/

total 28
drwxr-xr-x 7 root root 4096 Mar  2 22:14 .
drwxr-xr-x 4 root root 4096 Mar  2 22:14 ..
drwxr-xr-x 6 root root 4096 Mar  2 22:14 .locks
drwxr-xr-x 5 root root 4096 Mar  2 22:14 models--KittenML--kitten-tts-micro-0.8
drwxr-xr-x 5 root root 4096 Mar  2 22:14 models--KittenML--kitten-tts-mini-0.8
drwxr-xr-x 5 root root 4096 Mar  2 22:14 models--KittenML--kitten-tts-nano-0.8-fp32
drwxr-xr-x 5 root root 4096 Mar  2 22:14 models--KittenML--kitten-tts-nano-0.8-int8


## 3.Helper: Generate & Play Audio

A utility function that generates speech, measures inference time, and plays the audio inline.

In [19]:
import IPython.display as ipd
import numpy as np

SAMPLE_RATE = 24000

def generate_and_play(model, model_label, text, voice):
    """Generate TTS audio, print timing stats, and play inline."""
    t0 = time.time()
    audio = model.generate(text, voice=voice)
    elapsed = time.time() - t0

    duration = len(audio) / SAMPLE_RATE
    rtf = elapsed / duration if duration > 0 else float('inf')

    print(f"🔊 {model_label} | Voice: {voice}")
    print(f"   Inference: {elapsed:.2f}s | Audio: {duration:.2f}s | RTF: {rtf:.2f}x")
    display(ipd.Audio(audio, rate=SAMPLE_RATE, autoplay=False))
    print()

## 4.Compare generated audios from all four models

In [20]:
text = "Upon the edge of silicon and wire, a quiet spark ignites a silent fire. It does not need the cloud to see the light, but calculates the world within the night. Through tangled nodes and weights so neatly spun, the tiny model sees the rising sun"

for label, model in models.items():
    generate_and_play(model, label, text, voice="Jasper")

🔊 Nano int8 (15M params, ~20MB) ⭐ | Voice: Jasper
   Inference: 17.21s | Audio: 15.49s | RTF: 1.11x



🔊 Nano fp32 (15M params, ~50MB) | Voice: Jasper
   Inference: 4.98s | Audio: 15.59s | RTF: 0.32x



🔊 Micro (40M params, ~40MB) | Voice: Jasper
   Inference: 12.95s | Audio: 15.42s | RTF: 0.84x



🔊 Mini (80M params, ~79MB) | Voice: Jasper
   Inference: 24.54s | Audio: 13.94s | RTF: 1.76x


## 5.Print onnx graph and focuse on the Nano FP32 and INT8 models

### 5a. Install ONNX and print model graph info

In [21]:
# install onnx
! pip install onnx

In [22]:
# helper function for printing model graph info
import onnx
def print_graph(model_path):
    print(f"Loading model from {model_path}...")
    model = onnx.load(model_path)

    print("\n--- MODEL INPUTS ---")
    for input_tensor in model.graph.input:
        print(f"Name: {input_tensor.name}")
        # Extract the shape
        shape = [dim.dim_value for dim in input_tensor.type.tensor_type.shape.dim]
        print(f"Shape: {shape}")

    print("\n--- MODEL OUTPUTS ---")
    for output_tensor in model.graph.output:
        print(f"Name: {output_tensor.name}")
        shape = [dim.dim_value for dim in output_tensor.type.tensor_type.shape.dim]
        print(f"Shape: {shape}")

    # print every single node/layer
    # print(onnx.helper.printable_graph(model.graph))

In [23]:
# let's only focuse on kitten-tts-nano-0.8-int8 and kitten-tts-nano-0.8-fp32
# which are the first two models of the list
# because we will do the quantization analysis for the nano modesl
# for label, tts_wrapper in list(models.items())[:2]:
for label, tts_wrapper in models.items():
    print(f"\n{'='*40}")
    print(f"Inspecting: {label}")
    print(f"{'='*40}")

    # Extract the absolute file path from inside the wrapper object
    # (tts_wrapper is the KittenTTS class, .model is the ONNX class, .model_path is the string)
    cached_file_path = tts_wrapper.model.model_path

    # Now pass the actual string path to your function
    print_graph(cached_file_path)


Inspecting: Nano int8 (15M params, ~20MB) ⭐
Loading model from /root/.cache/huggingface/hub/models--KittenML--kitten-tts-nano-0.8-int8/snapshots/84781d74e29ee25217551556398b42f80593a813/kitten_tts_nano_v0_8.onnx...

--- MODEL INPUTS ---
Name: input_ids
Shape: [1, 0]
Name: style
Shape: [1, 256]
Name: speed
Shape: [1]

--- MODEL OUTPUTS ---
Name: waveform
Shape: [0]
Name: duration
Shape: [0]

Inspecting: Nano fp32 (15M params, ~50MB)
Loading model from /root/.cache/huggingface/hub/models--KittenML--kitten-tts-nano-0.8-fp32/snapshots/7a1db645b1f3ab9420761d87428e042b9cec3f26/kitten_tts_nano_v0_8.onnx...

--- MODEL INPUTS ---
Name: input_ids
Shape: [1, 0]
Name: style
Shape: [1, 256]
Name: speed
Shape: [1]

--- MODEL OUTPUTS ---
Name: waveform
Shape: [0]
Name: duration
Shape: [0]

Inspecting: Micro (40M params, ~40MB)
Loading model from /root/.cache/huggingface/hub/models--KittenML--kitten-tts-micro-0.8/snapshots/1ccf72b2c2048fd17efac7de2fab32d10e225084/kitten_tts_micro_v0_8.onnx...

--- MO

### 5b. Extract weights/tensors from the model.

In [24]:
from onnx import numpy_helper
import numpy as np

def extract_weights(model_path):
    """Extracts initializers (weights) from an ONNX model into a dictionary."""
    model = onnx.load(model_path)
    weights = {}
    for init in model.graph.initializer:
        # Convert ONNX tensor to standard Numpy array
        weights[init.name] = numpy_helper.to_array(init)
    return weights

INT8_weights = extract_weights(list(models.items())[0][1].model.model_path )
FP32_weights = extract_weights(list(models.items())[1][1].model.model_path)

In [25]:
print("Length of the INT8 model:", len(INT8_weights))
print("Length of the FP32 model:", len(FP32_weights))

Length of the INT8 model: 1010
Length of the FP32 model: 698


In [26]:
# lets see the layer names for the FP32 model
print('Names of the FP32 model tensors:')
print(list(FP32_weights.keys()))

Names of the FP32 model tensors:
['kmodel.bert.embeddings.word_embeddings.weight', 'kmodel.bert.embeddings.position_embeddings.weight', 'kmodel.bert.embeddings.token_type_embeddings.weight', 'kmodel.bert.embeddings.LayerNorm.weight', 'kmodel.bert.embeddings.LayerNorm.bias', 'kmodel.bert.encoder.embedding_hidden_mapping_in.bias', 'kmodel.bert.encoder.albert_layer_groups.0.albert_layers.0.full_layer_layer_norm.weight', 'kmodel.bert.encoder.albert_layer_groups.0.albert_layers.0.full_layer_layer_norm.bias', 'kmodel.bert.encoder.albert_layer_groups.0.albert_layers.0.attention.query.bias', 'kmodel.bert.encoder.albert_layer_groups.0.albert_layers.0.attention.key.bias', 'kmodel.bert.encoder.albert_layer_groups.0.albert_layers.0.attention.value.bias', 'kmodel.bert.encoder.albert_layer_groups.0.albert_layers.0.attention.dense.bias', 'kmodel.bert.encoder.albert_layer_groups.0.albert_layers.0.attention.LayerNorm.weight', 'kmodel.bert.encoder.albert_layer_groups.0.albert_layers.0.attention.LayerNor

In [27]:
# Find common layers to compare
import numpy as np
common_layers = set(FP32_weights.keys()).intersection(set(INT8_weights.keys()))
print("We need to find the common layers between the two models:")
print(f"Found {len(common_layers)} common layers between INT8 and FP32 models.")
# print(common_layers)

We need to find the common layers between the two models:
Found 571 common layers between INT8 and FP32 models.


Here is the mathematical reason **why the INT8 model has a larger number of layers than the FP32 model**.

What does **571 common layers** mean in the first place?

That is the number of layers inside the INT8 model that have not been quantized to INT8 or you could say those layers are untouched, which means they are kept as FP32.

Becuase **we only quantized layers such as Convolution, Linear, MatMul, and LSTM layers** and layers only contian Biases, LayerNorms, and Constants are kept in high precision.

Compressing those would lead to degradation of the audio quality while saving almost zero disk space.



The mathematical breakdown of the layer numbers:

In onnx model, when converting from FP32 to INT8, **1 FP32 layer -> 3 INT8 tensors (weight_quantized, weight_scale, weight_zero_point).** You can check model structure by using [Netron](https://netron.app/).

Recall that the FP32 model has a total of 698 layers, that means the **quantized layers** are calcuated as the number of total layers subtract the untouched layers, which are **698-571 = 127** layers.

These **127 quantized layers are then expended to 127 * 3 = 381** after quantization and these layers are kept inside the INT8 model.

So, there are now 571 + 381 = 952 layers inside the INT8 model.

But what about the remaining 1010 - 952 = 58 layers?

There is a tensor called "Conv_output_0_bias_reshape_shape", ONNX would automatically inject into your model when you quantize a Convolutional (Conv) layer.
It is used when you want to match a 1D array ([Bias]) to a 3D block ([Batch, Channels, Time]). So this is not important at all.

However, let's check if we indeed have 58 of those layers inside the INT8 model!



In [28]:
# lets check for the bias_reshape_layers
bias_reshape_layers = []

# Loop through all 1010 keys in the INT8 model
for key in INT8_weights.keys():
    # Filter for the exact suffix the ONNX exporter added to the Convolutions
    if "Conv_output_0_bias_reshape_shape" in key:
        bias_reshape_layers.append(key)

# Sort them alphabetically for a clean output
bias_reshape_layers.sort()

print(f"Found exactly {len(bias_reshape_layers)} Bias Reshape Constants in the INT8 model:\n")
print("-" * 60)

for i, layer_name in enumerate(bias_reshape_layers, 1):
    # Print the count alongside the layer name
    print(f"{i:02d}. {layer_name}")

Found exactly 58 Bias Reshape Constants in the INT8 model:

------------------------------------------------------------
01. /N.0/conv1/Conv_output_0_bias_reshape_shape
02. /N.0/conv2/Conv_output_0_bias_reshape_shape
03. /N.1/conv1/Conv_output_0_bias_reshape_shape
04. /N.1/conv2/Conv_output_0_bias_reshape_shape
05. /N.2/conv1/Conv_output_0_bias_reshape_shape
06. /N.2/conv2/Conv_output_0_bias_reshape_shape
07. /decoder/asr_res/asr_res.0/Conv_output_0_bias_reshape_shape
08. /decoder/decode.0/conv1/Conv_output_0_bias_reshape_shape
09. /decoder/decode.0/conv2/Conv_output_0_bias_reshape_shape
10. /decoder/decode.1/conv1/Conv_output_0_bias_reshape_shape
11. /decoder/decode.1/conv2/Conv_output_0_bias_reshape_shape
12. /decoder/decode.2/conv1/Conv_output_0_bias_reshape_shape
13. /decoder/decode.2/conv2/Conv_output_0_bias_reshape_shape
14. /decoder/decode.3/conv1/Conv_output_0_bias_reshape_shape
15. /decoder/decode.3/conv2/Conv_output_0_bias_reshape_shape
16. /decoder/encode/conv1/Conv_output_0

### 5c. Calcuation of Cosine Similarity between FP32 and INT8 models for diagnosing the INT8 model.


Cosine similarity measures the cosine of the angle between two vectors projected in a multi-dimensional space.
It determines how similar the two vectors are in terms of their direction, \regardless of their **magnitude.**

For two flattened weight tensors (or activation tensors), $A$ (FP32) and $B$ (INT8), the formula is:

$$\text{Cosine Similarity} = \frac{A \cdot B}{\|A\|\|B\|} = \frac{\sum_{i=1}^{n} A_i B_i}{\sqrt{\sum_{i=1}^{n} A_i^2} \sqrt{\sum_{i=1}^{n} B_i^2}}$$


The result ranges from **-1 to 1:**


*   1.0: Perfect alignment (identical directions)
*   0.0: Orthogonal (completely uncorrelated).
*   -1.0: Completely opposed (worst).

In quantization, you generally want scores above 0.99 for weights.

 Anything lower usually indicates a layer that is highly sensitive to quantization and might need to be kept in FP16 or FP32.

When you calculate the cosine similarity, you cannot directly compare raw INT8 integers to FP32 floats because cosine similarity is sensitive to shifts (the zero-point).

Before calculating the similarity, you must simulate the hardware's process and dequantize the INT8 weights back into a dequantized FP32 format using their specific scale and zero-point parameters:

$$\text{Weight}_{\text{FP32}} = (\text{Weight}_{\text{INT8}} - \text{ZeroPoint}) \times \text{Scale}$$

Let's use the dequantized equation for now, the quantization and dequantization will be another topic for model optimization in deep learning. Which I will cover next time.


A simple example of how those two equations work:

1. Imagine a layer in your text-to-speech model has the FP32 fromat tensor:$A = [2.05, 0.95, -1.10].$

2. The correspoding INT8 tensor can be calcuated by using the quantization equation: $[21, 10, -11], $Scale: $0.1$, Zero-Point: $0$

3. The dequantization formula as listed above: $(\text{Raw} - \text{ZeroPoint}) \times \text{Scale}$.

*    $B = ([21, 10, -11] - 0) \times 0.1$
*    $B = [2.10, 1.00, -1.10]$
*    Notice how $B$ is slightly different from $A$ because we lost precision during the 8-bit rounding process ($2.05 \to 2.10$).

4. Now we calcuate the cosine similarity of $A$ and $B$.

*    The Dot Product ($A \cdot B$): Multiply them together element-by-element and add them up.

*    $(2.05 \times 2.10) + (0.95 \times 1.00) + (-1.10 \times -1.10) = 4.305 + 0.95 + 1.21 = \mathbf{6.465}$

*    **The Magnitude of A ($\|A\|$)**: $\sqrt{2.05^2 + 0.95^2 + (-1.10)^2} = \mathbf{2.513}$

*    **The Magnitude of B** ($\|B\|$):$\sqrt{2.10^2 + 1.00^2 + (-1.10)^2} = \mathbf{2.573}$

*    Finally, we divide the Dot Product by the two Magnitudes multiplied together:

$$\text{Similarity} = \frac{6.465}{2.513 \times 2.573} = \frac{6.465}{6.4659} = \mathbf{0.9998}$$

A score of **0.9998 is above the 0.99 threshold,** this tells us there is barely an accuracy drop because of this layer/tensor inside the quantized model.

When the **quantized model has obvious accuracy drop or performance degradation** and we want to diagnose which layers/tensors are resposible for it.
The **Cosine Similarity report is the perfect thing to check with**, we may consider use the FP32 or FP16 weight for the layer if the score is less than 0.99.



In [29]:
def calculate_similarities(INT8_weights, FP32_weights):
  """Calcuates the similarity for each layer between INT8 and FP32 models."""
  similarities = {}
  twoD_transposed_count = 0
  threeD_transposed_count = 0
  twoD_hidden_transposed_count = 0
  threeD_hidden_transposed_count = 0

  quantized_layer_count = 0

  print("🔜 Let's begining the comparison\n")

  for fp32_name, fp32_w in FP32_weights.items():
      # 1. Check if it's an exact match (like a bias or constant)
      if fp32_name in INT8_weights:
          # get the INT8 weight values
          int8_w = INT8_weights[fp32_name]

      # 2. Check if it was renamed to a quantized format
      else:
          # Standard PyTorch layers ending in .weight
          if fp32_name.endswith(".weight"):
              # right to left split
              base_name = fp32_name.rsplit(".weight", 1)[0]
              # print(base_name)
              q_name = f"{base_name}.weight_quantized"
              scale_name = f"{base_name}.weight_scale"
              zp_name = f"{base_name}.weight_zero_point"

          # ONNX internal layers (like onnx::LSTM_5652)
          else:
              base_name = fp32_name
              q_name = f"{base_name}_quantized"
              scale_name = f"{base_name}_scale"
              zp_name = f"{base_name}_zero_point"

          # If the quantized version exists, grab all three parts (applies to both cases above)
          if q_name in INT8_weights:
              raw_int8 = INT8_weights[q_name]
              scale = INT8_weights[scale_name]
              zero_point = INT8_weights[zp_name]

              # If scale is 1D (per-channel) but the weights are 2D/3D,
              # we pad the scale's shape with 1s so it aligns properly.
              if scale.ndim == 1 and raw_int8.ndim > 1:
                  # Creates a shape like (2, 1, 1) or (128, 1) depending on the tensor
                  target_shape = [-1] + [1] * (raw_int8.ndim - 1)
                  # print(target_shape)
                  scale = scale.reshape(target_shape)
                  zero_point = zero_point.reshape(target_shape)

              # Dequantize back to simulated FP32
              int8_w = (raw_int8.astype(np.float32) - zero_point) * scale

              quantized_layer_count += 1
          else:
              # If we STILL can't find it, skip to the next layer
              continue

      # --- THE SHAPE CHECK & AUTO-REPAIR ---
      # if the shape of FP32 and INT8 are not equal, transpose it and check again
      if fp32_w.shape != int8_w.shape:
          # Check for 2D if the dimensions are simply reversed (transposed)
          # (N, M) -> (M, N)
          if len(fp32_w.shape) == 2 and fp32_w.shape == int8_w.shape[::-1]:
              # Suppressing the print statement here to keep the log clean
              print(f"transposed detected: {fp32_name} | FP32: {fp32_w.shape} vs INT8: {int8_w.shape}")

              # Transpose the int8 weights so they align for the 1D flattening
              int8_w = int8_w.T
              twoD_transposed_count += 1

          # Check for 3D tensors (LSTMs) where axes 1 and 2 are swapped
          # (0, 1, 2) -> (0, 2, 1)
          elif len(fp32_w.shape) == 3 and fp32_w.shape == (int8_w.shape[0], int8_w.shape[2], int8_w.shape[1]):
              print(f"transposed detected: {fp32_name} | FP32: {fp32_w.shape} vs INT8: {int8_w.shape}")
              int8_w = np.transpose(int8_w, (0, 2, 1))
              threeD_transposed_count += 1

          else:
              # If dimensions don't match and aren't transposed, we can't compare them
              print(f"CRITICAL MISMATCH: {fp32_name} | FP32: {fp32_w.shape} vs INT8: {int8_w.shape}")
              continue

      # SMART TRANSPOSE & MATH (Use float64 to prevent overflow)
      v1 = fp32_w.flatten().astype(np.float64)
      v2 = int8_w.flatten().astype(np.float64)

      # Calculate norms
      norm_v1 = np.linalg.norm(v1)
      norm_v2 = np.linalg.norm(v2)

      # Calculate standard similarity
      similarity = np.dot(v1, v2) / (norm_v1 * norm_v2 + 1e-8)

      # --- THE HIDDEN TRANSPOSE CHECK (2D & 3D) ---
      # If it's a matrix with a terrible score, test if it's a hidden transposed square
      # before, we only transpose when we see, take 2D tensor for an example:
      # (N, M) vs (M, N) but what if the tensors are (N, N) and (N, N)?
      # if the tensors are both (N, N), and we do not transpose them
      # the similarity will be really small, so we need to fix this:
      if similarity < 0.5:
          sim_transposed = 0.0

          # Flatten the transposed version and re-calcuate on similarity
          if len(fp32_w.shape) == 2:
              v2_transposed = int8_w.T.flatten().astype(np.float64)
              sim_transposed = np.dot(v1, v2_transposed) / (norm_v1 * norm_v2 + 1e-8)
              twoD_hidden_transposed_count += 1
          elif len(fp32_w.shape) == 3:
              v2_transposed = np.transpose(int8_w, (0, 2, 1)).flatten().astype(np.float64)
              sim_transposed = np.dot(v1, v2_transposed) / (norm_v1 * norm_v2 + 1e-8)
              threeD_hidden_transposed_count += 1

          # if the transposed version is above the threshhold after re-calculation,
          # we keep it
          if sim_transposed > 0.99:
              print(f"hidden transpose detected: {fp32_name}  | FP32: {fp32_w.shape} vs INT8: {int8_w.shape}")
              similarity = sim_transposed

      # Catch NaNs from zero-arrays
      if np.isnan(similarity):
          similarity = 0.0

      similarities[fp32_name] = similarity

      if similarity < 0.98:
          # Suppress the print for constants/biases that are naturally zero
          # in ONNX models, some biases and constants are literally initialized as arrays of pure zeros.
          # When the math script tries to compare an array of zeros to another array of zeros,
          # it triggers a divide-by-zero failsafe and returns 0.0.
          # it is not a real error, so we told the script to ignore them.
          # so, the biases and constants will not sure in the POOR SIMILARITY
          if similarity < 0.0001 and ("Constant" in fp32_name or "bias" in fp32_name):
              continue
          print(f"⚠️POOR SIMILARITY [{similarity:.4f}]: {fp32_name}")

  print(f"\n📊 Total comparison: {len(similarities)} layers.")
  print(f"🗜️ Total quantized layers compared: {quantized_layer_count}")
  print(f"🔄 2D realigned {twoD_transposed_count} transposed matrices.")
  print(f"🧊 3D realigned {threeD_transposed_count} transposed matrices.")
  print(f"🕵️‍♂️ 2D hidden realigned {twoD_hidden_transposed_count} transposed matrices.")
  print(f"🧩 3D hidden realigned {threeD_hidden_transposed_count} transposed matrices.")
  return similarities

In [30]:
similarities = calculate_similarities(INT8_weights, FP32_weights)

🔜 Let's begining the comparison

transposed detected: kmodel.predictor.text_encoder.lstms.1.fc.weight | FP32: (256, 128) vs INT8: (128, 256)
transposed detected: kmodel.predictor.text_encoder.lstms.3.fc.weight | FP32: (256, 128) vs INT8: (128, 256)
transposed detected: kmodel.predictor.F0.0.norm1.fc.weight | FP32: (256, 128) vs INT8: (128, 256)
transposed detected: kmodel.predictor.F0.0.norm2.fc.weight | FP32: (256, 128) vs INT8: (128, 256)
transposed detected: kmodel.predictor.F0.1.norm1.fc.weight | FP32: (256, 128) vs INT8: (128, 256)
hidden transpose detected: kmodel.predictor.F0.1.norm2.fc.weight  | FP32: (128, 128) vs INT8: (128, 128)
hidden transpose detected: kmodel.predictor.F0.2.norm1.fc.weight  | FP32: (128, 128) vs INT8: (128, 128)
hidden transpose detected: kmodel.predictor.F0.2.norm2.fc.weight  | FP32: (128, 128) vs INT8: (128, 128)
transposed detected: kmodel.predictor.N.0.norm1.fc.weight | FP32: (256, 128) vs INT8: (128, 256)
transposed detected: kmodel.predictor.N.0.nor

The above code is how we calcuate the cosine similarity, When the ONNX exporter optimizes a model for CPU inference, it often rotates (transposes) certain weight matrices to make the math run faster.

So we need to transpose it back when calculating, this is why there are so many **"transposed detected"** and **"hidden transpose detected"** in the print statement.

I print out 2D, 3D and hidden transposes along with their tensor shapes.

If you are interested in how and why do I do those, see the comments in the calculate_similarities function.

Finally, the poor similarity layers are only:

**⚠️POOR SIMILARITY [0.0000]: onnx::Range_3985**

**⚠️POOR SIMILARITY [0.0000]: onnx::Expand_3997**

**⚠️POOR SIMILARITY [0.0000]: onnx::Slice_746**

They are not important at all because they are not learnable weights and they are not even inside the quantized layers.

**This means the kitten-tts-nano-0.8-int8 model has no performance loss compared to kitten-tts-nano-0.8-fp32 model**

Let's save the whole report to a text file, at the end you will get

Total near-zero layers found: 10

Except from the 3 POOR SIMILARITY above, the other 7  are all bias and constant which are again not learnable weights, so again, the
**kitten-tts-nano-0.8-int8 model has no performance loss compared to kitten-tts-nano-0.8-fp32 model**

### 5d. Save the similarity report to text file.

In [31]:
def save_similarity_scores(similarities, filename="layer_similarities_report.txt"):
    """Saves all layer similarity scores and near-zero alerts to a text file."""

    with open(filename, 'w') as f:
        # --- Write all layers and their similarity scores ---
        f.write("="*50 + "\n")
        f.write("LAYER SIMILARITY SCORES\n")
        f.write("="*50 + "\n")
        for name, score in similarities.items():
            f.write(f"{name}: {score:.4f}\n")

        # --- Write layers with near-zero similarity ---
        f.write("\n" + "="*50 + "\n")
        f.write("ZERO OR NEAR-ZERO SIMILARITY LAYERS (< 1e-4)\n")
        f.write("="*50 + "\n")

        # Filter the dictionary for scores less than 0.0001
        near_zero_layers = {name: score for name, score in similarities.items() if score < 1e-4}

        if near_zero_layers:
            for name, score in near_zero_layers.items():
                # Using extended decimals to see exactly how small it is
                f.write(f"{name}: {score:.8f}\n")
            f.write(f"\nTotal near-zero layers found: {len(near_zero_layers)}\n")
        else:
            f.write("None found! All layers have a similarity score higher than 0.0001.\n")

    # Print a single clean confirmation to the Colab output
    print(f"📄✨ Success! Layer similarity report safely exported to '{filename}' 💾📉")

In [32]:
# To run it, just call:
save_similarity_scores(similarities)

📄✨ Success! Layer similarity report safely exported to 'layer_similarities_report.txt' 💾📉


The code implementation and mathematical analysis in this project were developed in collaboration with Google Gemini.